In [3]:
import os
import numpy as np
import pandas as pd
from scipy.signal import hilbert

# ==========================================
# 1. KONFIGURASI & MAPPING CHANNEL
# ==========================================
CHANNEL_NAMES = {
    0: "Fp1",
    1: "Fp2",
    2: "F3",
    3: "F4",
    4: "C3",
    5: "C4",
    6: "O1",
    7: "O2",
}

def get_wpli_feature_names(channel_map):
    names = []
    n_ch = len(channel_map)
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            names.append(f"{channel_map[i]}-{channel_map[j]}")
    return names

# Generate nama kolom fitur satu kali di awal
WPLI_FEATURE_NAMES = get_wpli_feature_names(CHANNEL_NAMES)

# ==========================================
# 2. FUNGSI INTI KOMPUTASI wPLI
# ==========================================
def compute_wpli_window(window_data):
    """Menghitung weighted Phase Lag Index (wPLI) antar channel
    menggunakan Hilbert Transform (kebal zero-lag / volume conduction).

    window_data : np.array [n_channels, n_samples]
    """
    window_data = np.asarray(window_data, dtype=float)
    n_ch, _ = window_data.shape

    # Transformasi Hilbert untuk mendapatkan analytic signal, magnitudo, & fase
    analytic = hilbert(window_data, axis=1)
    amps = np.abs(analytic)
    phase = np.angle(analytic)

    wpli_vals = []
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            diff_phase = phase[i] - phase[j]
            # Komponen imaginer dari cross-spectrum berbobot magnitudo: A_i * A_j * sin(phi_i - phi_j)
            imag_cross = amps[i] * amps[j] * np.sin(diff_phase)

            num = np.abs(np.mean(imag_cross))
            den = np.mean(np.abs(imag_cross))

            # Cegah pembagian dengan nol
            wpli = 0.0 if den == 0 else num / den
            wpli_vals.append(wpli)

    return np.array(wpli_vals, dtype=np.float32)

# ==========================================
# 3. PROSES EKSTRAKSI PER FILE (wPLI AR)
# ==========================================
def process_eeg_wpli(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    files = sorted([f for f in os.listdir(input_folder) if f.endswith(".csv")])
    print(f"🚀 Memulai ekstraksi wPLI (aktif-reflektif). Ditemukan {len(files)} file.")

    dp_cols = [f"datapoint{i}" for i in range(1, 251)]

    for i, fname in enumerate(files, start=1):
        # Deteksi label Aktif / Reflektif
        if "aktif" in fname.lower():
            current_label = "aktif"
        elif "reflektif" in fname.lower():
            current_label = "reflektif"
        else:
            current_label = "unknown"

        csv_path = os.path.join(input_folder, fname)
        df = pd.read_csv(csv_path)
        rows_out = []

        grouped = df.groupby(["nomor", "window"])

        for (trial_id, window_id), g in grouped:
            if g.shape[0] != 8:
                continue

            g = g.sort_values("channel")
            window_data = g[dp_cols].values

            # Hitung wPLI
            wpli_feat = compute_wpli_window(window_data)

            res_row = {
                "subject": i,
                "trial": trial_id,
                "window": window_id,
                "label": current_label,
            }

            for fname_feat, val in zip(WPLI_FEATURE_NAMES, wpli_feat):
                res_row[fname_feat] = val

            rows_out.append(res_row)

        df_final = pd.DataFrame(rows_out)
        out_filename = f"subject_{i:02d}_wPLI_AR.csv"
        df_final.to_csv(os.path.join(output_folder, out_filename), index=False)

        print(f"✅ Selesai: {out_filename} | Total Window: {len(df_final)} | Label: {current_label}")

# ==========================================
# 4. EKSEKUSI
# ==========================================
INPUT_PATH = "data-awal-aktif-reflektif-new"
OUTPUT_PATH = "features-wpli-ar"

process_eeg_wpli(INPUT_PATH, OUTPUT_PATH)

🚀 Memulai ekstraksi wPLI (aktif-reflektif). Ditemukan 18 file.
✅ Selesai: subject_01_wPLI_AR.csv | Total Window: 300 | Label: aktif
✅ Selesai: subject_02_wPLI_AR.csv | Total Window: 300 | Label: aktif
✅ Selesai: subject_03_wPLI_AR.csv | Total Window: 300 | Label: aktif
✅ Selesai: subject_04_wPLI_AR.csv | Total Window: 300 | Label: aktif
✅ Selesai: subject_05_wPLI_AR.csv | Total Window: 300 | Label: aktif
✅ Selesai: subject_06_wPLI_AR.csv | Total Window: 300 | Label: aktif
✅ Selesai: subject_07_wPLI_AR.csv | Total Window: 300 | Label: aktif
✅ Selesai: subject_08_wPLI_AR.csv | Total Window: 300 | Label: aktif
✅ Selesai: subject_09_wPLI_AR.csv | Total Window: 300 | Label: aktif
✅ Selesai: subject_10_wPLI_AR.csv | Total Window: 300 | Label: reflektif
✅ Selesai: subject_11_wPLI_AR.csv | Total Window: 300 | Label: reflektif
✅ Selesai: subject_12_wPLI_AR.csv | Total Window: 300 | Label: reflektif
✅ Selesai: subject_13_wPLI_AR.csv | Total Window: 300 | Label: reflektif
✅ Selesai: subject_14_wP

In [4]:
import pandas as pd
import os
import glob

# 1. Konfigurasi Folder
INPUT_FOLDER = "features-wpli-ar"
OUTPUT_FILE = "combined-FE-wPLI-VV.csv"

def merge_subject_files(input_path, output_name):
    # Mengambil semua list file .csv di folder
    all_files = glob.glob(os.path.join(input_path, "*.csv"))
    
    # Sort file agar berurutan dari subject_01 sampai subject_14
    all_files.sort()
    
    if not all_files:
        print(f"❌ Tidak ada file CSV ditemukan di folder '{input_path}'")
        return

    print(f"📂 Menemukan {len(all_files)} file. Memulai penggabungan...")

    # List untuk menampung dataframe
    df_list = []

    for filename in all_files:
        df = pd.read_csv(filename)
        df_list.append(df)
        print(f"✅ Menambahkan: {os.path.basename(filename)}")

    # Menggabungkan semua dataframe menjadi satu
    df_combined = pd.concat(df_list, ignore_index=True)

    # Menyimpan hasil gabungan
    df_combined.to_csv(output_name, index=False)
    
    print("-" * 30)
    print(f"🚀 BERHASIL! File gabungan disimpan sebagai: {output_name}")
    print(f"📊 Total baris keseluruhan: {len(df_combined)}")

# 2. Eksekusi
merge_subject_files(INPUT_FOLDER, OUTPUT_FILE)

📂 Menemukan 18 file. Memulai penggabungan...
✅ Menambahkan: subject_01_wPLI_AR.csv
✅ Menambahkan: subject_02_wPLI_AR.csv
✅ Menambahkan: subject_03_wPLI_AR.csv
✅ Menambahkan: subject_04_wPLI_AR.csv
✅ Menambahkan: subject_05_wPLI_AR.csv
✅ Menambahkan: subject_06_wPLI_AR.csv
✅ Menambahkan: subject_07_wPLI_AR.csv
✅ Menambahkan: subject_08_wPLI_AR.csv
✅ Menambahkan: subject_09_wPLI_AR.csv
✅ Menambahkan: subject_10_wPLI_AR.csv
✅ Menambahkan: subject_11_wPLI_AR.csv
✅ Menambahkan: subject_12_wPLI_AR.csv
✅ Menambahkan: subject_13_wPLI_AR.csv
✅ Menambahkan: subject_14_wPLI_AR.csv
✅ Menambahkan: subject_15_wPLI_AR.csv
✅ Menambahkan: subject_16_wPLI_AR.csv
✅ Menambahkan: subject_17_wPLI_AR.csv
✅ Menambahkan: subject_18_wPLI_AR.csv
------------------------------
🚀 BERHASIL! File gabungan disimpan sebagai: combined-FE-wPLI-VV.csv
📊 Total baris keseluruhan: 5400
